# Adversarial Attacks & Defences: CNN (ResNet-20) vs. Vision Transformer (ViT) on CIFAR-10

One-click notebook: **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

Pipeline: pretrained clean CNN/ViT -> native PyTorch FGSM & PGD -> 10-epoch AMP adversarial fine-tuning
(eps = 8/255) -> accuracy-vs-eps curves, cost of adversarial training, transferability, example grid.
Estimated total time on a T4: roughly 20-45 minutes (not yet measured; see README).

**Optional GitHub push:** add a Colab secret named `GITHUB_TOKEN` (fine-grained token, *Contents: read/write*
on the repo only). With it, checkpoints, CSV/MD tables and PNG plots are pushed after every stage and every 5
training epochs. Without it everything still runs, nothing is pushed.

In [ ]:
# ---- Configuration -------------------------------------------------------------------------------
import base64, os, subprocess

REPO_URL = "https://github.com/Mehtab404/DeepLearningProject.git"
MOUNT_DRIVE = True          # Colab only: keep checkpoints on Google Drive so a disconnect loses nothing
RUN_QUICK_TEST = False      # True = tiny smoke test first (a few minutes on GPU)
AUTO_PUSH = True            # only has an effect when a GITHUB_TOKEN secret is found
DRIVE_DIR = "/content/drive/MyDrive/adv_cnn_vs_vit_ckpts"

# ---- Optional GITHUB_TOKEN (Colab secret / Kaggle secret / environment variable) -------------------
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
if not GITHUB_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass

# The token is passed to git through environment variables only (never written to .git/config or logs).
if GITHUB_TOKEN:
    basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    os.environ.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                       "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"})
    print("GITHUB_TOKEN found: results will be pushed to GitHub.")
else:
    AUTO_PUSH = False
    print("No GITHUB_TOKEN found: running without GitHub pushes.")

# ---- Working directory = a clone of the repo (so pushes land in the right place) ------------------
BASE = "/content" if os.path.isdir("/content") else ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
REPO_DIR = os.path.join(BASE, "DeepLearningProject")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone failed (private repo without token?). Continuing in a fresh local folder, pushes disabled.")
        os.makedirs(REPO_DIR, exist_ok=True)
        subprocess.run(["git", "init", "-b", "main"], cwd=REPO_DIR, capture_output=True)
        AUTO_PUSH = False
os.chdir(REPO_DIR)
subprocess.run(["git", "config", "user.name", "colab-agent"])
subprocess.run(["git", "config", "user.email", "colab-agent@users.noreply.github.com"])
os.makedirs("src", exist_ok=True); os.makedirs("checkpoints", exist_ok=True); os.makedirs("outputs", exist_ok=True)
print("Working in", os.getcwd())

if MOUNT_DRIVE and os.path.isdir("/content"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped:", type(e).__name__)
        DRIVE_DIR = ""
else:
    DRIVE_DIR = ""

In [ ]:
!pip -q install timm transformers seaborn tqdm pandas
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (use --quick-test)")

In [ ]:
%%writefile src/__init__.py
# Adversarial robustness: CNN vs ViT on CIFAR-10

In [ ]:
%%writefile src/utils.py
"""Small shared helpers: seeding, device selection, JSON I/O, git auto-push, Drive sync."""
import json
import os
import random
import shutil
import subprocess
import sys

import numpy as np
import torch

SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _exclude_oversized_files(repo_dir, limit_mb=95):
    """GitHub rejects files > 100MB. Add any .pth/.pt above `limit_mb` to .git/info/exclude."""
    try:
        exclude = os.path.join(repo_dir, ".git", "info", "exclude")
        if not os.path.isdir(os.path.dirname(exclude)):
            return
        existing = open(exclude, encoding="utf-8").read() if os.path.exists(exclude) else ""
        new = []
        for root, _dirs, files in os.walk(os.path.join(repo_dir, "checkpoints")):
            for f in files:
                p = os.path.join(root, f)
                if f.endswith((".pth", ".pt")) and os.path.getsize(p) > limit_mb * 1024 * 1024:
                    rel = os.path.relpath(p, repo_dir).replace(os.sep, "/")
                    if rel not in existing:
                        new.append(rel)
        if new:
            with open(exclude, "a", encoding="utf-8") as fh:
                fh.write("\n" + "\n".join(new) + "\n")
            print(f"[AUTO-PUSH] Excluded from git (>{limit_mb}MB): {', '.join(new)}")
    except OSError:
        pass


def git_auto_push(message: str, repo_dir: str = ".") -> bool:
    """Python equivalent of scripts/auto_push.sh. Never raises: training must not die on a git error.

    Stages everything, commits if there is something to commit, and pushes HEAD to origin.
    """

    def run(*cmd):
        return subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True)

    try:
        if run("git", "rev-parse", "--is-inside-work-tree").returncode != 0:
            print("[AUTO-PUSH] Not a git repository, skipping.")
            return False
        _exclude_oversized_files(repo_dir)
        run("git", "add", ".")
        if run("git", "diff", "--cached", "--quiet").returncode == 0:
            print("[AUTO-PUSH] No file changes to commit.")
            return True
        commit = run("git", "commit", "-m", message)
        if commit.returncode != 0:
            print(f"[AUTO-PUSH] Commit failed: {commit.stderr.strip()[:300]}")
            return False
        push = run("git", "push", "origin", "HEAD")
        if push.returncode != 0:
            # stderr can echo the remote URL; never print it in full.
            print("[AUTO-PUSH] Push failed (check remote / token). Commit kept locally.")
            return False
        print(f"[AUTO-PUSH] Successfully pushed to GitHub: {message}")
        return True
    except Exception as e:  # noqa: BLE001
        print(f"[AUTO-PUSH] Skipped due to error: {type(e).__name__}")
        return False


def sync_to_drive(src_dir: str, drive_dir: str):
    """Copy a directory tree (e.g. checkpoints/) to Google Drive so a disconnect never loses work."""
    if not drive_dir:
        return
    try:
        os.makedirs(drive_dir, exist_ok=True)
        for name in os.listdir(src_dir):
            s = os.path.join(src_dir, name)
            if os.path.isfile(s):
                shutil.copy2(s, os.path.join(drive_dir, name))
    except Exception as e:  # noqa: BLE001
        print(f"[DRIVE] Sync skipped: {type(e).__name__}: {e}", file=sys.stderr)


def restore_from_drive(dest_dir: str, drive_dir: str):
    """Copy files from Drive into dest_dir if they are not already there."""
    if not drive_dir or not os.path.isdir(drive_dir):
        return
    os.makedirs(dest_dir, exist_ok=True)
    for name in os.listdir(drive_dir):
        d = os.path.join(dest_dir, name)
        s = os.path.join(drive_dir, name)
        if os.path.isfile(s) and not os.path.exists(d):
            shutil.copy2(s, d)
            print(f"[DRIVE] Restored {name}")

In [ ]:
%%writefile src/models.py
"""Model definitions: normalization wrapper + pretrained CIFAR-10 CNN and ViT loaders.

Golden rule: every tensor that reaches an attack lives in [0, 1]. The (x - mean) / std step
(and bilinear upsampling for the ViT) happens INSIDE forward(), so epsilon and clamp(0, 1)
are exact in pixel space for every architecture.
"""
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

HF_VIT_ID = "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10"
HUB_REPO = "chenyaofo/pytorch-cifar-models:master"


def _with_retries(fn, what="download", attempts=5, base_delay=5):
    """Run fn(); retry transient network failures (HTTP 5xx, timeouts) with a growing delay."""
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:  # noqa: BLE001 - urllib/requests/hub raise many different error types
            if i == attempts:
                raise
            wait = base_delay * i
            print(f"[retry] {what} failed ({type(e).__name__}: {str(e)[:120]}); attempt {i}/{attempts}, "
                  f"retrying in {wait}s")
            time.sleep(wait)


class NormalizedModelWrapper(nn.Module):
    def __init__(self, model, mean=CIFAR_MEAN, std=CIFAR_STD, img_size=None):
        super().__init__()
        self.model = model
        self.img_size = img_size
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        if self.img_size is not None and x.shape[-1] != self.img_size:
            x = F.interpolate(x, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
        x_norm = (x - self.mean) / self.std
        out = self.model(x_norm)
        return out.logits if hasattr(out, 'logits') else out


def load_cnn_clean(device, pretrained=True, arch="cifar10_resnet20"):
    """CNN-Clean: pretrained CIFAR-10 ResNet from chenyaofo/pytorch-cifar-models (no training needed).

    pretrained=False builds the same architecture without downloading weights (used when a local
    checkpoint will be loaded on top).
    """
    # ":master" pins the branch. Without it torch.hub first probes github.com/<repo>/tree/main/ to guess
    # the default branch, and a transient GitHub 5xx on that probe crashed a full Colab run.
    base = _with_retries(lambda: torch.hub.load(HUB_REPO, arch, pretrained=pretrained, trust_repo=True),
                         what="torch.hub " + arch)
    model = NormalizedModelWrapper(base, CIFAR_MEAN, CIFAR_STD, img_size=None)
    return model.to(device).eval()


def load_vit_clean(device, backbone="timm-tiny", pretrained=True):
    """ViT-Clean.

    backbone="timm-tiny": timm vit_tiny_patch16_224 (ImageNet weights, new 10-class head).
        Returns needs_clean_finetune=True -> caller runs a short clean fine-tune (default 5 epochs).
    backbone="hf-base":   HuggingFace ViT-Base already fine-tuned on CIFAR-10. No clean training,
        but ~16x more compute per image than ViT-Tiny, so adversarial fine-tuning is much slower.

    Returns (model, needs_clean_finetune).
    """
    # Both ViTs expect 224x224 input normalised with mean=std=0.5.
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    if backbone == "timm-tiny":
        import timm
        base = _with_retries(
            lambda: timm.create_model("vit_tiny_patch16_224", pretrained=pretrained, num_classes=10),
            what="timm vit_tiny weights")
        needs_ft = pretrained  # when pretrained=False we only build the shell for a local checkpoint
    elif backbone == "hf-base":
        from transformers import AutoConfig, ViTForImageClassification
        if pretrained:
            base = _with_retries(lambda: ViTForImageClassification.from_pretrained(HF_VIT_ID),
                                 what="HF ViT-Base weights")
        else:
            base = ViTForImageClassification(AutoConfig.from_pretrained(HF_VIT_ID))
        needs_ft = False
    else:
        raise ValueError(f"Unknown ViT backbone: {backbone}")
    model = NormalizedModelWrapper(base, mean, std, img_size=224)
    return model.to(device).eval(), needs_ft


def save_checkpoint(model, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    torch.save(model.state_dict(), path)


def load_checkpoint(model, path, device):
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()


def sanity_check(device=None):
    """Task 2.4: both models must output (batch, 10) on a dummy [0, 1] tensor of shape (4, 3, 32, 32)."""
    device = device or torch.device("cpu")
    x = torch.rand(4, 3, 32, 32, device=device)
    cnn = load_cnn_clean(device)
    vit, _ = load_vit_clean(device)
    with torch.no_grad():
        for name, m in (("CNN", cnn), ("ViT", vit)):
            out = m(x)
            assert out.shape == (4, 10), f"{name} output shape {tuple(out.shape)} != (4, 10)"
            print(f"[sanity] {name}: input {tuple(x.shape)} -> output {tuple(out.shape)} OK")


if __name__ == "__main__":
    sanity_check()

In [ ]:
%%writefile src/attacks.py
"""Native PyTorch L-infinity attacks: FGSM, PGD and RS-FGSM.

References
  [1] Goodfellow et al., Explaining and Harnessing Adversarial Examples, ICLR 2015.
  [2] Madry et al., Towards Deep Learning Models Resistant to Adversarial Attacks, ICLR 2018.
  [3] Wong et al., Fast is better than free, ICLR 2020 (RS-FGSM).

Conventions
  * x is always in [0, 1]; normalisation happens inside the model's forward().
  * Attacks never change model.train()/eval() mode. The caller decides (use model.eval()).
  * Gradients are taken with torch.autograd.grad, so no .grad is accumulated on model parameters.
  * amp=True runs the forward/backward under autocast (CUDA only). The loss is summed and computed in
    fp32, which keeps fp16 input-gradients from underflowing to 0 (sign() is scale invariant).
"""
import torch
import torch.nn.functional as F


def _input_grad(model, x, y, amp=False):
    """Gradient of the summed cross-entropy loss w.r.t. the input x."""
    x = x.clone().detach().requires_grad_(True)
    use_amp = bool(amp) and x.is_cuda
    with torch.autocast(device_type="cuda" if x.is_cuda else "cpu", enabled=use_amp):
        logits = model(x)
    loss = F.cross_entropy(logits.float(), y, reduction="sum")
    grad = torch.autograd.grad(loss, x)[0]
    return torch.nan_to_num(grad.detach())


def fgsm_attack(model, x, y, eps, amp=False):
    """x_adv = clip_[0,1]( x + eps * sign( grad_x L(theta, x, y) ) )   [Goodfellow et al., 2015]"""
    if eps == 0:
        return x.clone().detach()
    grad = _input_grad(model, x, y, amp)
    x_adv = x.detach() + eps * grad.sign()
    return torch.clamp(x_adv, 0.0, 1.0).detach()


def pgd_attack(model, x, y, eps, alpha=None, steps=10, random_start=True, amp=False):
    """L-inf PGD with projection onto the eps-ball around x and the [0, 1] box   [Madry et al., 2018]

    x^0     = clip_[0,1]( x + U(-eps, eps) )   (or x if random_start=False)
    x^{t+1} = clip_[0,1]( x + clip_[-eps,eps]( x^t + alpha * sign(grad) - x ) )
    Default step size: alpha = max(eps / 4, 1/255).
    """
    if eps == 0:
        return x.clone().detach()
    if alpha is None:
        alpha = max(eps / 4.0, 1.0 / 255.0)
    x = x.detach()
    if random_start:
        x_adv = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)
    else:
        x_adv = x.clone()
    for _ in range(steps):
        grad = _input_grad(model, x_adv, y, amp)
        x_adv = x_adv + alpha * grad.sign()
        delta = torch.clamp(x_adv - x, -eps, eps)      # project onto the L-inf ball
        x_adv = torch.clamp(x + delta, 0.0, 1.0)       # project onto the valid image box
    return x_adv.detach()


def rs_fgsm_attack(model, x, y, eps, alpha=None, amp=False):
    """Random-start FGSM ("Fast AT", Wong et al., 2020): U(-eps, eps) init + one step of size alpha.

    Default alpha = 1.25 * eps, as recommended in the paper.
    """
    if eps == 0:
        return x.clone().detach()
    if alpha is None:
        alpha = 1.25 * eps
    x = x.detach()
    x0 = torch.clamp(x + torch.empty_like(x).uniform_(-eps, eps), 0.0, 1.0)
    grad = _input_grad(model, x0, y, amp)
    delta = torch.clamp(x0 + alpha * grad.sign() - x, -eps, eps)
    return torch.clamp(x + delta, 0.0, 1.0).detach()


def check_perturbation(x, x_adv, eps, tol=1e-6):
    """Assert the L-inf budget and the [0, 1] pixel range (task 3.4)."""
    linf = torch.max(torch.abs(x_adv - x)).item()
    assert linf <= eps + tol, f"L-inf violated: {linf:.8f} > eps {eps:.8f}"
    assert 0.0 <= x_adv.min().item() and x_adv.max().item() <= 1.0, "x_adv left the [0, 1] range"
    return linf


def self_test(device=None, n=8):
    """Unit assertions on a tiny random CNN. Run with: python -m src.attacks"""
    import torch.nn as nn
    device = device or torch.device("cpu")
    torch.manual_seed(0)
    model = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                          nn.Flatten(), nn.Linear(8, 10)).to(device).eval()
    x = torch.rand(n, 3, 32, 32, device=device)
    y = torch.randint(0, 10, (n,), device=device)
    for eps in (0.0, 1 / 255, 4 / 255, 8 / 255, 16 / 255):
        for name, fn in (("fgsm", lambda: fgsm_attack(model, x, y, eps)),
                         ("pgd10", lambda: pgd_attack(model, x, y, eps, steps=10)),
                         ("pgd-no-rs", lambda: pgd_attack(model, x, y, eps, steps=3, random_start=False)),
                         ("rs_fgsm", lambda: rs_fgsm_attack(model, x, y, eps))):
            x_adv = fn()
            linf = check_perturbation(x, x_adv, eps)
            if eps == 0:
                assert linf == 0.0, f"{name}: eps=0 must return x unchanged"
        print(f"[attacks] eps={eps * 255:.0f}/255: fgsm / pgd / rs_fgsm all satisfy L-inf <= eps and [0, 1]")
    assert all(p.grad is None for p in model.parameters()), "attacks must not accumulate parameter grads"
    print("[attacks] self-test passed")


if __name__ == "__main__":
    self_test()

In [ ]:
%%writefile src/data.py
"""CIFAR-10 data loading. Tensors stay in [0, 1] (ToTensor only); normalisation lives in the model."""
import os
import shutil
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from .utils import SEED

CIFAR_TAR = "cifar-10-python.tar.gz"


def prepare_data(root="data", drive_dir=None, attempts=3):
    """Make sure CIFAR-10 is on disk, with retries and an optional Google Drive cache.

    The official host can be very slow (a Colab run measured ~60 kB/s, i.e. ~47 min for 170 MB), so the
    archive is cached in <drive_dir>/dataset/ and restored from there on later runs.
    """
    dst = os.path.join(root, CIFAR_TAR)
    cached = os.path.join(drive_dir, "dataset", CIFAR_TAR) if drive_dir else None
    os.makedirs(root, exist_ok=True)
    if cached and not os.path.exists(dst) and os.path.exists(cached):
        shutil.copy2(cached, dst)
        print("[data] restored CIFAR-10 archive from Drive cache")
    for i in range(1, attempts + 1):
        try:
            datasets.CIFAR10(root=root, train=True, download=True)
            datasets.CIFAR10(root=root, train=False, download=True)
            break
        except Exception as e:  # noqa: BLE001
            if i == attempts:
                raise
            print(f"[retry] CIFAR-10 download failed ({type(e).__name__}); attempt {i}/{attempts}")
            time.sleep(10 * i)
    if cached and os.path.exists(dst) and not os.path.exists(cached):
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        shutil.copy2(dst, cached)
        print("[data] cached CIFAR-10 archive to Drive")


def get_train_loader(batch_size=128, root="data", max_samples=0, augment=True, num_workers=2, seed=SEED):
    tfms = []
    if augment:
        tfms += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]
    tfms.append(transforms.ToTensor())  # NO Normalize here (golden rule)
    ds = datasets.CIFAR10(root=root, train=True, download=True, transform=transforms.Compose(tfms))
    if max_samples and max_samples < len(ds):
        g = np.random.RandomState(seed)
        idx = g.permutation(len(ds))[:max_samples].tolist()
        ds = Subset(ds, idx)
    pin = torch.cuda.is_available()
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                      pin_memory=pin, drop_last=True,
                      persistent_workers=num_workers > 0)


def stratified_indices(targets, n, seed=SEED):
    """n // 10 images per class, drawn with a fixed seed. Returns a sorted index list."""
    targets = np.asarray(targets)
    classes = np.unique(targets)
    per_class = max(1, n // len(classes))
    rng = np.random.RandomState(seed)
    idx = []
    for c in classes:
        cls_idx = np.where(targets == c)[0]
        idx.extend(rng.choice(cls_idx, size=min(per_class, len(cls_idx)), replace=False).tolist())
    return sorted(idx)


def get_eval_tensors(n=1000, root="data", seed=SEED):
    """Fixed stratified test subset as in-memory tensors (X in [0,1], shape (N,3,32,32); Y int64)."""
    ds = datasets.CIFAR10(root=root, train=False, download=True, transform=transforms.ToTensor())
    idx = stratified_indices(ds.targets, n, seed)
    X = torch.stack([ds[i][0] for i in idx])
    Y = torch.tensor([ds.targets[i] for i in idx], dtype=torch.long)
    return X, Y

In [ ]:
%%writefile src/defense.py
"""Fast adversarial fine-tuning (10 epochs, AMP) + optional short clean fine-tune for the timm ViT.

Per batch:
  1. model.eval(), build x_adv (PGD-5 or RS-FGSM) under autocast.
  2. model.train(), loss = 0.5 * CE(f(x), y) + 0.5 * CE(f(x_adv), y).
     Clean and adversarial halves go through ONE forward pass on the concatenated batch [x; x_adv].
     With equal-sized halves, the mean CE over the 2B samples is exactly the 50/50 loss above.
  3. GradScaler step.
Checkpoints, history JSON and a resumable training-state file are written after every epoch.
"""
import os
import time

import torch
import torch.nn.functional as F
from tqdm import tqdm

from .attacks import pgd_attack, rs_fgsm_attack
from .models import save_checkpoint
from .utils import git_auto_push, load_json, save_json, sync_to_drive


def _autocast(device, enabled):
    return torch.autocast(device_type=device.type, enabled=bool(enabled) and device.type == "cuda")


def make_train_attack(method, eps):
    if method == "pgd5":
        return lambda m, x, y: pgd_attack(m, x, y, eps=eps, alpha=2 / 255, steps=5, random_start=True, amp=True)
    if method == "rs_fgsm":
        return lambda m, x, y: rs_fgsm_attack(m, x, y, eps=eps, alpha=1.25 * eps, amp=True)
    raise ValueError(f"method must be 'pgd5' or 'rs_fgsm', got {method!r}")


def update_history(history_path, tag, entry=None, **meta):
    hist = load_json(history_path, {}) or {}
    rec = hist.setdefault(tag, {"epochs": []})
    rec.update(meta)
    if entry is not None:
        rec["epochs"] = [e for e in rec["epochs"] if e["epoch"] != entry["epoch"]] + [entry]
    save_json(hist, history_path)


def is_training_complete(history_path, tag, epochs):
    rec = (load_json(history_path, {}) or {}).get(tag, {})
    return bool(rec.get("completed")) and len(rec.get("epochs", [])) >= epochs


def adversarial_finetune(model, train_loader, epochs=10, eps=8 / 255, method="pgd5", lr=1e-4,
                         device=None, tag="cnn_adv", ckpt_dir="checkpoints", drive_dir=None,
                         auto_push_every=0, push_fn=git_auto_push, max_batches=0):
    """Adversarially fine-tune `model` in place and return the history list.

    max_batches > 0 limits batches per epoch (smoke tests only).
    Resumes automatically from checkpoints/<tag>_state.pt if a previous run was interrupted.
    """
    device = device or next(model.parameters()).device
    model.to(device)
    attack = make_train_attack(method, eps)
    use_amp = device.type == "cuda"

    ckpt_path = os.path.join(ckpt_dir, f"{tag}.pth")
    state_path = os.path.join(ckpt_dir, f"{tag}_state.pt")
    hist_path = os.path.join(ckpt_dir, "training_history.json")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch = 0
    if os.path.exists(state_path):
        st = torch.load(state_path, map_location=device)
        model.load_state_dict(st["model"])
        optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"])
        scaler.load_state_dict(st["scaler"])
        start_epoch = st["epoch"]
        print(f"[{tag}] Resuming from epoch {start_epoch}/{epochs}")

    update_history(hist_path, tag, completed=False, method=method, eps=eps, lr=lr, target_epochs=epochs,
                   amp=use_amp, device=str(device))

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        tot_loss = tot_clean = tot_adv = n = 0
        pbar = tqdm(train_loader, desc=f"[{tag}] Epoch {epoch + 1}/{epochs}", leave=False)
        for b, (x, y) in enumerate(pbar):
            if max_batches and b >= max_batches:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            model.eval()
            with _autocast(device, use_amp):
                x_adv = attack(model, x, y)

            model.train()
            optimizer.zero_grad(set_to_none=True)
            with _autocast(device, use_amp):
                logits = model(torch.cat([x, x_adv], dim=0))
            logits = logits.float()
            bs = x.size(0)
            loss = 0.5 * F.cross_entropy(logits[:bs], y) + 0.5 * F.cross_entropy(logits[bs:], y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            tot_loss += loss.item() * bs
            tot_clean += (logits[:bs].argmax(1) == y).sum().item()
            tot_adv += (logits[bs:].argmax(1) == y).sum().item()
            n += bs
            pbar.set_postfix(loss=f"{tot_loss / n:.3f}", clean=f"{100 * tot_clean / n:.1f}%",
                             robust=f"{100 * tot_adv / n:.1f}%")
        pbar.close()
        scheduler.step()
        model.eval()

        entry = {"epoch": epoch + 1, "train_loss": tot_loss / max(n, 1),
                 "train_clean_acc": 100 * tot_clean / max(n, 1), "train_robust_acc": 100 * tot_adv / max(n, 1),
                 "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0}
        print(f"[{tag}] Epoch {entry['epoch']}/{epochs} | loss {entry['train_loss']:.4f} | "
              f"clean {entry['train_clean_acc']:.2f}% | robust {entry['train_robust_acc']:.2f}% | "
              f"{entry['seconds']:.0f}s")

        # checkpoint after EVERY epoch
        save_checkpoint(model, ckpt_path)
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch + 1}, state_path)
        update_history(hist_path, tag, entry=entry)
        sync_to_drive(ckpt_dir, drive_dir)
        if auto_push_every and (epoch + 1) % auto_push_every == 0 and (epoch + 1) < epochs:
            push_fn(f"chore(train): {tag} epoch {epoch + 1}/{epochs} checkpoint")

    update_history(hist_path, tag, completed=True)
    if os.path.exists(state_path):
        os.remove(state_path)  # training finished; the resume state is no longer needed
    sync_to_drive(ckpt_dir, drive_dir)
    model.eval()
    return load_json(hist_path)[tag]["epochs"]


def clean_finetune(model, train_loader, epochs=5, lr=2e-4, device=None, max_batches=0, desc="ViT-clean"):
    """Plain (non-adversarial) fine-tune; used once to adapt the ImageNet timm ViT-Tiny to CIFAR-10."""
    device = device or next(model.parameters()).device
    use_amp = device.type == "cuda"
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for epoch in range(epochs):
        model.train()
        tot = correct = n = 0
        pbar = tqdm(train_loader, desc=f"[{desc}] Epoch {epoch + 1}/{epochs}", leave=False)
        for b, (x, y) in enumerate(pbar):
            if max_batches and b >= max_batches:
                break
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with _autocast(device, use_amp):
                logits = model(x)
            loss = F.cross_entropy(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tot += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
            pbar.set_postfix(loss=f"{tot / n:.3f}", acc=f"{100 * correct / n:.1f}%")
        pbar.close()
        scheduler.step()
        print(f"[{desc}] Epoch {epoch + 1}/{epochs} | loss {tot / max(n, 1):.4f} | acc {100 * correct / max(n, 1):.2f}%")
    return model.eval()

In [ ]:
%%writefile src/evaluate.py
"""Evaluation: epsilon sweep (FGSM / PGD-10), cross-model transferability, result tables.

All attacks run in full fp32 (no autocast) so the reported numbers are not affected by fp16 noise.
"""
import os

import pandas as pd
import torch

from .attacks import fgsm_attack, pgd_attack

EPS_LIST = [0, 1, 2, 4, 8, 16]            # in units of 1/255
MODEL_ORDER = ["CNN-Clean", "ViT-Clean", "CNN-AdvTrained", "ViT-AdvTrained"]


@torch.no_grad()
def _correct(model, x, y):
    return (model(x).argmax(1) == y).sum().item()


def accuracy(model, X, Y, device, batch_size=100):
    model.eval()
    c = 0
    for i in range(0, len(X), batch_size):
        x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)
        c += _correct(model, x, y)
    return 100.0 * c / len(X)


def _attack_fn(name, eps, pgd_steps):
    if name == "FGSM":
        return lambda m, x, y: fgsm_attack(m, x, y, eps)
    if name == "PGD-10":
        # alpha = max(eps/4, 1/255), 10 steps, random start (the evaluation defaults from the spec)
        return lambda m, x, y: pgd_attack(m, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps,
                                          random_start=True)
    raise ValueError(name)


def evaluate_epsilon_sweep(models, X, Y, device, eps_list=EPS_LIST, pgd_steps=10, batch_size=100, seed=42):
    """White-box accuracy of every model vs. epsilon under FGSM and PGD. Returns a long-format DataFrame."""
    rows = []
    for name, model in models.items():
        model.eval()
        for attack in ("FGSM", "PGD-10"):
            for e in eps_list:
                torch.manual_seed(seed)                      # reproducible random starts
                eps = e / 255.0
                fn = _attack_fn(attack, eps, pgd_steps)
                correct = 0
                for i in range(0, len(X), batch_size):
                    x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)
                    x_adv = fn(model, x, y)
                    correct += _correct(model, x_adv, y)
                acc = 100.0 * correct / len(X)
                rows.append({"model": name, "attack": attack, "eps_255": e, "accuracy": acc})
                print(f"[eval] {name:15s} {attack:6s} eps={e:2d}/255 -> {acc:5.1f}%")
    return pd.DataFrame(rows)


def evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=10, batch_size=100, seed=42):
    """matrix.loc[source, target] = accuracy of `target` on PGD examples crafted against `source`.

    The diagonal is the white-box accuracy. Adversarial examples are generated once per source model.
    """
    names = list(models.keys())
    mat = pd.DataFrame(index=names, columns=names, dtype=float)
    for src in names:
        torch.manual_seed(seed)
        models[src].eval()
        adv_batches = []
        for i in range(0, len(X), batch_size):
            x, y = X[i:i + batch_size].to(device), Y[i:i + batch_size].to(device)
            adv_batches.append(pgd_attack(models[src], x, y, eps, alpha=max(eps / 4, 1 / 255),
                                          steps=pgd_steps, random_start=True).cpu())
        X_adv = torch.cat(adv_batches)
        for tgt in names:
            mat.loc[src, tgt] = accuracy(models[tgt], X_adv, Y, device, batch_size)
        print(f"[transfer] source={src:15s} -> " + ", ".join(f"{t}: {mat.loc[src, t]:.1f}%" for t in names))
    mat.index.name = "source \\ target"
    return mat


def to_markdown(df, floatfmt="{:.1f}"):
    """Minimal DataFrame -> GitHub markdown table (avoids the optional `tabulate` dependency)."""
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join([" --- "] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        cells = [str(idx)] + [floatfmt.format(v) if isinstance(v, float) else str(v) for v in row.values]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def export_results(sweep_df, transfer_df, out_dir, note=""):
    """Deliverable 5: results_summary.csv / .md (+ transferability.csv)."""
    os.makedirs(out_dir, exist_ok=True)
    sweep_df.to_csv(os.path.join(out_dir, "results_summary.csv"), index=False)
    wide = sweep_df.pivot_table(index=["attack", "model"], columns="eps_255", values="accuracy", sort=False)
    wide.columns = [f"eps={c}/255" for c in wide.columns]
    wide.index = [f"{a}, {m}" for a, m in wide.index]   # no '|' here: it would break the markdown table
    wide.index.name = "attack, model"
    md = ["# Results summary (accuracy %, white-box)", ""]
    if note:
        md += [f"> {note}", ""]
    md += [to_markdown(wide), "", "## Transferability: PGD-10 at eps=8/255 (rows = source, cols = target; accuracy %)", "",
           to_markdown(transfer_df), ""]
    with open(os.path.join(out_dir, "results_summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md))
    transfer_df.to_csv(os.path.join(out_dir, "transferability.csv"))
    print("\n".join(md))

In [ ]:
%%writefile src/visualize.py
"""The four publication-quality figures (300 DPI PNG)."""
import os

import matplotlib
matplotlib.use("Agg")  # headless-safe; notebooks display the saved PNGs explicitly
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from .attacks import pgd_attack
from .models import CIFAR10_CLASSES

DPI = 300
COLORS = {"CNN": "#1f77b4", "ViT": "#d62728"}


def _style(name):
    arch = name.split("-")[0]
    return COLORS[arch], ("--" if "Adv" in name else "-"), ("s" if "Adv" in name else "o")


def _tag(fig, tag):
    if tag:
        fig.text(0.5, 0.005, tag, ha="center", va="bottom", fontsize=8, color="gray")


def plot_acc_vs_epsilon(df, path, tag=""):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
    for ax, attack in zip(axes, ("FGSM", "PGD-10")):
        for name in df["model"].unique():
            d = df[(df.model == name) & (df.attack == attack)].sort_values("eps_255")
            c, ls, mk = _style(name)
            ax.plot(d.eps_255, d.accuracy, color=c, linestyle=ls, marker=mk, linewidth=2, label=name)
        ax.set_title(f"{attack} attack")
        ax.set_xlabel(r"$\epsilon$ (in /255 units)")
        ax.set_xticks(sorted(df.eps_255.unique()))
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_ylim(0, 100)
    axes[1].legend(loc="upper right")
    fig.suptitle("Accuracy vs. perturbation budget on CIFAR-10 (solid = clean, dashed = adversarially fine-tuned)")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    _tag(fig, tag)
    fig.savefig(path, dpi=DPI)
    plt.close(fig)


def _acc(df, model, attack, eps):
    return float(df[(df.model == model) & (df.attack == attack) & (df.eps_255 == eps)].accuracy.iloc[0])


def plot_tradeoff(df, path, tag=""):
    metrics = [("Clean (eps=0)", "FGSM", 0), ("FGSM (eps=8/255)", "FGSM", 8), ("PGD-10 (eps=8/255)", "PGD-10", 8)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, arch in zip(axes, ("CNN", "ViT")):
        before = [_acc(df, f"{arch}-Clean", a, e) for _, a, e in metrics]
        after = [_acc(df, f"{arch}-AdvTrained", a, e) for _, a, e in metrics]
        xs = np.arange(len(metrics))
        w = 0.36
        ax.bar(xs - w / 2, before, w, label="Before (clean model)", color="#9ecae1" if arch == "CNN" else "#fcae91")
        ax.bar(xs + w / 2, after, w, label="After 10-epoch adv. fine-tuning", color=COLORS[arch])
        for i in range(len(metrics)):
            delta = after[i] - before[i]
            ax.text(xs[i] - w / 2, before[i] + 1, f"{before[i]:.1f}", ha="center", fontsize=8)
            ax.text(xs[i] + w / 2, after[i] + 1, f"{after[i]:.1f}", ha="center", fontsize=8)
            label = r"$\Delta$Acc$_{clean}$" if i == 0 else r"$\Delta$Acc$_{robust}$"
            ax.text(xs[i], max(before[i], after[i]) + 8, f"{label} = {delta:+.1f} pp", ha="center", fontsize=9,
                    fontweight="bold", color="darkgreen" if delta >= 0 else "darkred")
        ax.set_xticks(xs)
        ax.set_xticklabels([m[0] for m in metrics])
        ax.set_title(f"{arch}: cost of adversarial training vs. robustness gain")
        ax.set_ylim(0, 115)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(loc="upper right", fontsize=8)
    axes[0].set_ylabel("Accuracy (%)")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    _tag(fig, tag)
    fig.savefig(path, dpi=DPI)
    plt.close(fig)


def plot_transferability(matrix, path, tag=""):
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(matrix.astype(float), annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, ax=ax,
                cbar_kws={"label": "Target accuracy (%) - lower = stronger attack"})
    ax.set_xlabel("Target model (evaluated on)")
    ax.set_ylabel("Source model (adversarial examples crafted on)")
    ax.set_title(r"Transferability of PGD-10 examples ($\epsilon$ = 8/255); diagonal = white-box")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    _tag(fig, tag)
    fig.savefig(path, dpi=DPI)
    plt.close(fig)


def plot_adversarial_examples(models, X, Y, device, path, eps=8 / 255, source="CNN-Clean", n=4,
                              pgd_steps=10, tag=""):
    """Column 1: clean x + label. Column 2: magnified perturbation. Column 3: x_adv + predictions of all models."""
    src = models[source].eval()
    # pick n images of distinct classes that the source model classifies correctly
    chosen, seen = [], set()
    with torch.no_grad():
        for i in range(len(X)):
            y = int(Y[i])
            if y in seen:
                continue
            if src(X[i:i + 1].to(device)).argmax(1).item() == y:
                chosen.append(i)
                seen.add(y)
            if len(chosen) == n:
                break
    if not chosen:
        chosen = list(range(min(n, len(X))))
    x = X[chosen].to(device)
    y = Y[chosen].to(device)
    torch.manual_seed(42)
    x_adv = pgd_attack(src, x, y, eps, alpha=max(eps / 4, 1 / 255), steps=pgd_steps, random_start=True)
    pert = ((x_adv - x + eps) / (2 * eps)).clamp(0, 1)

    preds = {}
    with torch.no_grad():
        for name, m in models.items():
            preds[name] = m.eval()(x_adv).argmax(1).cpu().tolist()

    rows = len(chosen)
    fig, axes = plt.subplots(rows, 3, figsize=(10.5, 3.1 * rows))
    axes = np.atleast_2d(axes)
    for r in range(rows):
        axes[r, 0].imshow(x[r].permute(1, 2, 0).cpu().numpy())
        axes[r, 0].set_title(f"Clean: {CIFAR10_CLASSES[int(y[r])]}", fontsize=10)
        axes[r, 1].imshow(pert[r].permute(1, 2, 0).cpu().numpy())
        axes[r, 1].set_title(r"PGD-10 perturbation, magnified $(x_{adv}-x+\epsilon)/2\epsilon$", fontsize=8)
        axes[r, 2].imshow(x_adv[r].permute(1, 2, 0).detach().cpu().numpy())
        lines = []
        for name in models:
            p = preds[name][r]
            ok = "OK" if p == int(y[r]) else "WRONG"
            lines.append(f"{name}: {CIFAR10_CLASSES[p]} [{ok}]")
        axes[r, 2].set_title(f"x_adv (eps=8/255, crafted on {source})\n" + "\n".join(lines), fontsize=7)
        for c in range(3):
            axes[r, c].axis("off")
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    _tag(fig, tag)
    fig.savefig(path, dpi=DPI)
    plt.close(fig)


def make_all_plots(sweep_df, transfer_df, models, X, Y, device, out_dir, pgd_steps=10, tag=""):
    os.makedirs(out_dir, exist_ok=True)
    paths = {
        "acc_vs_epsilon": os.path.join(out_dir, "acc_vs_epsilon.png"),
        "tradeoff": os.path.join(out_dir, "clean_vs_robust_tradeoff.png"),
        "transfer": os.path.join(out_dir, "transferability_heatmap.png"),
        "examples": os.path.join(out_dir, "adversarial_examples_grid.png"),
    }
    plot_acc_vs_epsilon(sweep_df, paths["acc_vs_epsilon"], tag)
    plot_tradeoff(sweep_df, paths["tradeoff"], tag)
    plot_transferability(transfer_df, paths["transfer"], tag)
    plot_adversarial_examples(models, X, Y, device, paths["examples"], pgd_steps=pgd_steps, tag=tag)
    for p in paths.values():
        print(f"[plots] saved {p}")
    return paths

In [ ]:
%%writefile main.py
"""CLI entry point: CNN vs ViT adversarial robustness on CIFAR-10.

  python main.py --quick-test            # CPU smoke test (64 test images, 1 tiny epoch), no GPU needed
  python main.py --full-run              # Colab/Kaggle GPU: 10-epoch adversarial fine-tuning + full evaluation
  python main.py --epochs 10 --eval-samples 1000 --auto-push

Checkpoints in checkpoints/ are reused automatically: nothing that already exists is retrained.
"""
import argparse
import copy
import json
import os
import time

from src.data import get_eval_tensors, get_train_loader, prepare_data
from src.defense import adversarial_finetune, clean_finetune, is_training_complete
from src.evaluate import accuracy, evaluate_epsilon_sweep, evaluate_transferability, export_results
from src.models import load_checkpoint, load_cnn_clean, load_vit_clean, save_checkpoint
from src.utils import (SEED, get_device, git_auto_push, load_json, restore_from_drive, save_json, set_seed,
                       sync_to_drive)
from src.visualize import make_all_plots


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    mode = p.add_mutually_exclusive_group()
    mode.add_argument("--quick-test", action="store_true", help="CPU smoke test: 64 images, 1 short epoch")
    mode.add_argument("--full-run", action="store_true", help="Full GPU run (default when no mode is given)")
    p.add_argument("--epochs", type=int, default=10, help="adversarial fine-tuning epochs")
    p.add_argument("--eval-samples", type=int, default=1000, help="stratified test subset size")
    p.add_argument("--eps", type=float, default=8.0, help="training epsilon in /255 units")
    p.add_argument("--cnn-method", choices=["pgd5", "rs_fgsm"], default="pgd5")
    p.add_argument("--vit-method", choices=["pgd5", "rs_fgsm"], default="rs_fgsm",
                   help="rs_fgsm keeps ViT adversarial training inside the Colab T4 time budget")
    p.add_argument("--vit-backbone", choices=["timm-tiny", "hf-base"], default="timm-tiny")
    p.add_argument("--vit-clean-epochs", type=int, default=5, help="clean fine-tune epochs for timm ViT-Tiny")
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--max-train-samples", type=int, default=0, help="0 = full 50k training set")
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--data-root", default="data")
    p.add_argument("--ckpt-dir", default=None)
    p.add_argument("--out-dir", default="outputs")
    p.add_argument("--drive-dir", default=None, help="e.g. /content/drive/MyDrive/adv_ckpts (Colab)")
    p.add_argument("--force-retrain", action="store_true", help="ignore existing adversarial checkpoints")
    p.add_argument("--auto-push", action="store_true", help="git commit+push after each stage / every 5 epochs")
    return p.parse_args()


def clear_stale_quick_test_outputs(out_dir):
    """Quick-test artifacts are committed to git; delete them so they can never pass for full-run results."""
    info = load_json(os.path.join(out_dir, "run_info.json"), {}) or {}
    if info.get("mode") != "quick-test":
        return
    for name in ("acc_vs_epsilon.png", "clean_vs_robust_tradeoff.png", "transferability_heatmap.png",
                 "adversarial_examples_grid.png", "results_summary.csv", "results_summary.md",
                 "transferability.csv", "run_info.json"):
        p = os.path.join(out_dir, name)
        if os.path.exists(p):
            os.remove(p)
    print("[outputs] removed stale quick-test artifacts")


def main():
    args = parse_args()
    quick = args.quick_test
    device = get_device()
    set_seed(SEED)

    pgd_steps, max_batches, tag = 10, 0, ""
    if quick:
        args.epochs, args.eval_samples, args.batch_size = 1, 64, 16
        args.max_train_samples, args.vit_clean_epochs, args.num_workers = 64, 1, 0
        args.drive_dir = None   # never mix throw-away smoke-test checkpoints into the real Drive cache
        pgd_steps, max_batches = 2, 2
        tag = "QUICK-TEST smoke run (tiny N, 1 epoch, 2 PGD steps) - not a result"
    args.ckpt_dir = args.ckpt_dir or ("checkpoints_quick_test" if quick else "checkpoints")
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.out_dir, exist_ok=True)
    eps_train = args.eps / 255.0

    print(f"Device: {device} | mode: {'quick-test' if quick else 'full-run'} | ckpt: {args.ckpt_dir}")
    if device.type == "cpu" and not quick:
        print("WARNING: full-run on CPU will be extremely slow. Use Colab/Kaggle GPU (or --quick-test).")

    restore_from_drive(args.ckpt_dir, args.drive_dir)
    if not quick:
        clear_stale_quick_test_outputs(args.out_dir)
    push = (lambda msg: git_auto_push(msg)) if args.auto_push else (lambda msg: None)
    t_start = time.time()

    # Stage 1: clean baselines (pretrained, no/little training).
    # Models are loaded BEFORE the dataset so a network/hub failure shows up in seconds, not after a
    # long CIFAR-10 download.
    cnn_path = os.path.join(args.ckpt_dir, "cnn_clean.pth")
    vit_path = os.path.join(args.ckpt_dir, "vit_clean.pth")

    if os.path.exists(cnn_path):
        cnn_clean = load_checkpoint(load_cnn_clean(device, pretrained=False), cnn_path, device)
        print("[CNN-Clean] loaded from checkpoint")
    else:
        cnn_clean = load_cnn_clean(device)
        save_checkpoint(cnn_clean, cnn_path)

    vit_needs_ft = False
    if os.path.exists(vit_path):
        shell, _ = load_vit_clean(device, args.vit_backbone, pretrained=False)
        vit_clean = load_checkpoint(shell, vit_path, device)
        print("[ViT-Clean] loaded from checkpoint")
    else:
        vit_clean, vit_needs_ft = load_vit_clean(device, args.vit_backbone)

    prepare_data(args.data_root, args.drive_dir)
    train_loader = get_train_loader(args.batch_size, args.data_root, args.max_train_samples,
                                    num_workers=args.num_workers)

    if vit_needs_ft:
        print(f"[ViT-Clean] clean fine-tune on CIFAR-10 for {args.vit_clean_epochs} epochs")
        clean_finetune(vit_clean, train_loader, epochs=args.vit_clean_epochs, device=device,
                       max_batches=max_batches)
        save_checkpoint(vit_clean, vit_path)
    sync_to_drive(args.ckpt_dir, args.drive_dir)
    push("chore(train): clean CNN/ViT baseline checkpoints")

    # Stage 2: adversarial fine-tuning (skipped if already completed)
    hist_path = os.path.join(args.ckpt_dir, "training_history.json")
    adv_models = {}
    for tag_name, clean, method, label in (("cnn_adv", cnn_clean, args.cnn_method, "CNN-AdvTrained"),
                                           ("vit_adv", vit_clean, args.vit_method, "ViT-AdvTrained")):
        ckpt = os.path.join(args.ckpt_dir, f"{tag_name}.pth")
        model = copy.deepcopy(clean)
        if (os.path.exists(ckpt) and not args.force_retrain
                and is_training_complete(hist_path, tag_name, args.epochs)):
            load_checkpoint(model, ckpt, device)
            print(f"[{label}] completed checkpoint found -> skipping training")
        else:
            print(f"[{label}] adversarial fine-tuning: {args.epochs} epochs, {method}, eps={args.eps}/255")
            if args.force_retrain:
                for f in (ckpt, os.path.join(args.ckpt_dir, f"{tag_name}_state.pt")):
                    if os.path.exists(f):
                        os.remove(f)
            adversarial_finetune(model, train_loader, epochs=args.epochs, eps=eps_train, method=method,
                                 lr=args.lr, device=device, tag=tag_name, ckpt_dir=args.ckpt_dir,
                                 drive_dir=args.drive_dir, auto_push_every=5 if args.auto_push else 0,
                                 max_batches=max_batches)
            push(f"chore(train): {label} {args.epochs}-epoch adversarial fine-tuning finished")
        adv_models[label] = model.eval()

    models = {"CNN-Clean": cnn_clean, "ViT-Clean": vit_clean, **adv_models}
    for m in models.values():
        for prm in m.parameters():
            prm.requires_grad_(False)   # attacks need input grads only

    # Stage 3: evaluation
    X, Y = get_eval_tensors(args.eval_samples, args.data_root)
    print(f"Evaluation subset: {len(X)} stratified test images (seed {SEED})")
    sweep = evaluate_epsilon_sweep(models, X, Y, device, pgd_steps=pgd_steps)
    transfer = evaluate_transferability(models, X, Y, device, eps=8 / 255, pgd_steps=pgd_steps)
    note = (f"Quick-test smoke run only (N={len(X)}, {pgd_steps} PGD steps): not valid results."
            if quick else f"N={len(X)} stratified CIFAR-10 test images, seed {SEED}, PGD-{pgd_steps}.")
    export_results(sweep, transfer, args.out_dir, note)

    # Stage 4: figures
    make_all_plots(sweep, transfer, models, X, Y, device, args.out_dir, pgd_steps=pgd_steps, tag=tag)

    info = {"mode": "quick-test" if quick else "full-run", "device": str(device), "epochs": args.epochs,
            "eval_samples": len(X), "pgd_eval_steps": pgd_steps, "cnn_method": args.cnn_method,
            "vit_method": args.vit_method, "vit_backbone": args.vit_backbone,
            "clean_acc_full_subset": {k: accuracy(m, X, Y, device) for k, m in models.items()},
            "total_minutes": round((time.time() - t_start) / 60, 2)}
    save_json(info, os.path.join(args.out_dir, "run_info.json"))
    print(json.dumps(info, indent=2))
    push("results: evaluation tables and plots")
    print(f"Done in {info['total_minutes']} min. Outputs in {args.out_dir}/")


if __name__ == "__main__":
    main()

## Sanity checks

In [ ]:
# Task 2.4 / 3.4 sanity checks: model output shapes and attack budget/range assertions
from src.models import sanity_check
from src.attacks import self_test
sanity_check()
self_test()

## Optional quick smoke test

In [ ]:
if RUN_QUICK_TEST:
    !python main.py --quick-test

## Full run (train + evaluate + plots)

In [ ]:
# Full run: loads checkpoints/*.pth if they exist (Drive is restored first), otherwise trains.
extra = ""
if DRIVE_DIR: extra += f" --drive-dir '{DRIVE_DIR}'"
if AUTO_PUSH: extra += " --auto-push"
!python main.py --full-run --epochs 10 --eval-samples 1000 {extra}

## Results

In [ ]:
import json
from IPython.display import Image, Markdown, display
info = json.load(open("outputs/run_info.json"))
assert info["mode"] == "full-run", "outputs/ holds a quick-test run, not full-run results. Re-run the training cell."
for f in ["acc_vs_epsilon", "clean_vs_robust_tradeoff", "transferability_heatmap", "adversarial_examples_grid"]:
    display(Image(filename=f"outputs/{f}.png"))
display(Markdown(open("outputs/results_summary.md", encoding="utf-8").read()))

In [ ]:
from src.utils import git_auto_push
if AUTO_PUSH:
    git_auto_push("results(final): push complete CIFAR-10 CNN vs ViT robustness metrics and 300-DPI plots [Milestone 7]")